<a href="https://colab.research.google.com/github/sudhars97/Forum-Posts-code/blob/main/DL_M4_RR_C9_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import yfinance as ticker
from sklearn.preprocessing import StandardScaler

# 1. DATA ACQUISITION
print("Fetching SPY data...")
spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")
data = spy[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
data['Target'] = data['Close'].shift(-1)
data.dropna(inplace=True)

# Define periods
train_data, val_data, test_data = data['2004':'2021'], data['2022':'2024'], data['2025':'2026']

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
val_scaled = scaler.transform(val_data.drop('Target', axis=1))
test_scaled = scaler.transform(test_data.drop('Target', axis=1))

def create_sequences(data, target, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        # RNN Shape: (Sequence_Length, Input_Size)
        X.append(data[i:i+window])
        y.append(target.iloc[i+window])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

X_train, y_train = create_sequences(train_scaled, train_data['Target'])
X_val, y_val = create_sequences(val_scaled, val_data['Target'])
X_test, y_test = create_sequences(test_scaled, test_data['Target'])

# 2. RNN ARCHITECTURE (Section 9.4)
class FinancialRNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=64):
        super().__init__()
        # Captures dynamics via recurrent connections (Section 9.4.2)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # h_t = f(x_t, h_{t-1}) - info passes across time steps
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :]) # Use the last hidden state

# 3. TRAINING
model = FinancialRNN()
loss_fn, optimizer = nn.MSELoss(), torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Training RNN (Samples: {len(X_train)})...")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    # Gradient clipping is often used in RNNs to prevent exploding gradients (Section 9.5.3)
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}: Train Loss {loss.item():.2f}")

# 4. TESTING
model.eval()
with torch.no_grad():
    test_mse = loss_fn(model(X_test).squeeze(), y_test)
    print(f"\nFinal RNN Test MSE (2025-2026): {test_mse.item():.2f}")
    print("Execution complete. Temporal patterns processed.")

Fetching SPY data...


/tmp/ipykernel_631/3940338754.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_631/3940338754.py:19: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
/tmp/ipykernel_631/3940338754.py:20: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  val_scaled = scaler.transform(val_data.drop('Target', axis=1))
/tmp/ipykernel_631/3940338754.py:21: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  test_scaled = scaler.transform(test_data.drop('Target', axis=1))


Training RNN (Samples: 4522)...
Epoch 2: Train Loss 33638.81
Epoch 4: Train Loss 33605.15
Epoch 6: Train Loss 33567.69
Epoch 8: Train Loss 33524.31
Epoch 10: Train Loss 33472.92

Final RNN Test MSE (2025-2026): 394212.19
Execution complete. Temporal patterns processed.
